In [60]:
import pandas as pd
import numpy as np

In [62]:
df = pd.read_csv("MSW93_OrderMSW93_FamilyMSW93_GenusMSW93_Species.csv")

In [ ]:
df.head()

In [66]:
# Split the data into features (X) and target (y)
X = df.drop('12-2_Terrestriality', axis=1)
y = df['12-2_Terrestriality']

In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

In [ ]:
num_cols = [c for c in num_cols if c in X_train.columns]
cat_cols = [c for c in cat_cols if c in X_train.columns]
print("Filtered numeric:", num_cols)
print("Filtered categorical:", cat_cols)

Data Preprocessing Steps

Split Data for testing and training for Model 

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [127]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="median")
cat_imp = SimpleImputer(strategy="most_frequent")

X_train_num = num_imp.fit_transform(X_train[num_cols])
X_test_num  = num_imp.transform(X_test[num_cols])

X_train_cat = cat_imp.fit_transform(X_train[cat_cols])
X_test_cat  = cat_imp.transform(X_test[cat_cols])



In [129]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat_enc = ohe.fit_transform(X_train[cat_cols])
X_test_cat_enc  = ohe.transform(X_test[cat_cols])



scaler = StandardScaler()
X_train_num_s = scaler.fit_transform(X_train_num)
X_test_num_s  = scaler.transform(X_test_num)

import numpy as np
X_train_pre = np.hstack([scaler.fit_transform(X_train[num_cols]), X_train_cat_enc])
X_test_pre = np.hstack([scaler.transform(X_test[num_cols]), X_test_cat_enc])


In [131]:
cat_feature_names = ohe.get_feature_names_out(cat_cols)
all_features = num_cols + list(cat_feature_names)


Train Random Forest Classifier

In [134]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_pre, y_train)  


RandomForestClassifier(random_state=42)

Train and Get accuracy, confusion matrix of model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_pre, y_train)

y_pred = rf.predict(X_test_pre)
print("RF accuracy:", accuracy_score(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
print(X_train_pre.shape)  # Should output (n_samples, n_features)
print(len(y_train))      # Should output n_samples


In [ ]:
# Assuming these variables are defined:
# X_train_pre, X_test_pre — numpy arrays of transformed data
# ohe — fitted OneHotEncoder
# scaler — fitted StandardScaler
# num_cols, cat_cols — original column name lists

import pandas as pd

# Build feature names list
num_feature_names = num_cols
cat_feature_names = list(ohe.get_feature_names_out(cat_cols))
all_features = num_feature_names + cat_feature_names

# Create DataFrames with correct column names
df_train_pre = pd.DataFrame(X_train_pre, columns=all_features)
df_train_pre['12-2_Terrestriality'] = y_train.values

df_test_pre = pd.DataFrame(X_test_pre, columns=all_features)
df_test_pre['12-2_Terrestriality'] = y_test.values

# Save to CSV
df_train_pre.to_csv("preprocessed_train.csv", index=False)
df_test_pre.to_csv("preprocessed_test.csv", index=False)

print("Saved preprocessed_train.csv and preprocessed_test.csv")


Plot confusion Matrix

In [144]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:



from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

y_pred_rf = rf.predict(X_test_pre)
cm = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=rf.classes_, yticklabels=rf.classes_)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
import numpy as np

corr = df_train_pre[num_feature_names].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numeric Feature Correlation Matrix")
plt.show()


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

corr = df_train_pre[num_feature_names].corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.title("Correlation Matrix — Lower Triangle Only")
plt.show()

In [ ]:
threshold = 0.5
mask = np.abs(corr) < threshold

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.title(f"Correlations with |r| >= {threshold}")
plt.show()

Select features for correlation map plot

In [ ]:
subset = ['1-1_ActivityCycle', '5-1_AdultBodyMass_g', '8-1_AdultForearmLen_mm']  # example features
sub_corr = corr.loc[subset, subset]

plt.figure(figsize=(6, 5))
sns.heatmap(sub_corr, annot=True, fmt=".2f",
            cmap="coolwarm", center=0)
plt.title("Correlation of Selected Features")
plt.show()

Plot feature importance

In [ ]:
import pandas as pd

importances = rf.feature_importances_
feat_imp = pd.DataFrame({"feature": all_features, "importance": importances})
feat_imp = feat_imp.sort_values("importance", ascending=False).head(10)

sns.barplot(data=feat_imp, x="importance", y="feature")
plt.title("Top 10 Feature Importances (RF)")
plt.show()


Train Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize and train model
model = LinearRegression()
model.fit(X_train_pre, y_train)

Get accuracy for linear regression model

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions
y_pred = model.predict(X_test_pre)

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")


Train and get accuracy for the logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(solver="lbfgs", max_iter=1000)
model.fit(X_train_pre, y_train)
y_pred = model.predict(X_test_pre)
y_prob = model.predict_proba(X_test_pre)[:, 1]

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Make predictions
y_pred = model.predict(X_test_pre)

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

# Calculate R-squared
r2 = r2_score(y_test, y_pred)
print(f"R-squared: {r2}")
